# Notebook 09 — Urban Groundwater Pressure Index (UGPI)
**Stage 9 of the Ghana Groundwater–Subsidence Research Pipeline**

Constructs the Urban Groundwater Pressure Index (UGPI) for the 10 km analysis grid using six normalised indicators: building density, population density, built-up growth rate, drought frequency, GWSA trend, and a piped-water access proxy. Uses equal weighting as the baseline and performs sensitivity analysis with alternative weights.

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy.spatial import cKDTree
from scipy.stats import spearmanr

ROOT     = Path('.')
PROC_URB = ROOT / 'data' / 'processed' / 'urban_growth'
PROC_GW  = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
PROC_SUB = ROOT / 'data' / 'processed' / 'subsidence'
PROC_IDX = ROOT / 'data' / 'processed' / 'pressure_indices'
FIG_DIR  = ROOT / 'outputs' / 'figures'
TAB_DIR  = ROOT / 'outputs' / 'tables'
POP_DIR  = ROOT / 'data' / 'interim' / 'harmonized_grids' / 'population'

PROC_IDX.mkdir(parents=True, exist_ok=True)

TARGET_CRS = 'EPSG:32630'
print('Setup OK')


In [ ]:
# 2. Load 10km grid with urban indicators already computed
grid = gpd.read_file(PROC_URB / 'builtup_grid10km.gpkg')
print(f'Grid cells: {len(grid)}')
print(f'CRS: {grid.crs}')

# Compute cell centroids in WGS84 for spatial joins
grid_4326 = grid.to_crs('EPSG:4326')
grid['centroid_lon'] = grid_4326.geometry.centroid.x
grid['centroid_lat'] = grid_4326.geometry.centroid.y

# WorldPop population growth rate 2000-2020
pop_files = {2000:'gha_ppp_2000.tif', 2010:'gha_ppp_2010.tif', 2020:'gha_ppp_2020.tif'}
def raster_mean_by_cell(raster_path, gdf_utm):
    vals = []
    with rasterio.open(raster_path) as src:
        for geom in gdf_utm.geometry:
            try:
                out_img, _ = rio_mask(src, [geom.__geo_interface__], crop=True)
                data = out_img[0].astype(float)
                nd = src.nodata
                if nd is not None:
                    data[data == nd] = np.nan
                v = np.nansum(data)
                vals.append(v)
            except Exception:
                vals.append(np.nan)
    return vals

pop_path_2000 = POP_DIR / 'gha_ppp_2000.tif'
pop_path_2010 = POP_DIR / 'gha_ppp_2010.tif'
if pop_path_2000.exists() and pop_path_2010.exists():
    grid['pop_2000'] = raster_mean_by_cell(pop_path_2000, grid)
    grid['pop_2010'] = raster_mean_by_cell(pop_path_2010, grid)
    grid['pop_growth_rate'] = (
        (grid['population_2020'] - grid['pop_2000']) /
        (grid['pop_2000'] + 1)
    ).clip(0)
    print(f'Pop growth rate computed. Mean: {grid["pop_growth_rate"].mean():.2f}')
else:
    grid['pop_growth_rate'] = grid['pop_density_km2'] / (grid['pop_density_km2'].max() + 1)
    print('Using pop density as growth proxy')


In [ ]:
# 3. Spatially join GRACE-scale indicators to 10km grid (nearest-neighbour)

def join_grace_to_grid(nc_path, var_name, grid_df):
    ds = xr.open_dataset(nc_path)
    da = ds[var_name]
    lats = da.lat.values
    lons = da.lon.values
    vals = da.values
    ds.close()

    # Build KD-tree from GRACE pixel centres
    grace_pts = np.column_stack([
        np.repeat(lats, len(lons)),
        np.tile(lons, len(lats))
    ])
    grace_vals = vals.flatten()
    tree = cKDTree(grace_pts)

    # Query nearest GRACE pixel for each 10km cell centroid
    grid_pts = np.column_stack([grid_df['centroid_lat'].values,
                                 grid_df['centroid_lon'].values])
    dist, idx = tree.query(grid_pts, k=1)
    return grace_vals[idx]

# Drought frequency (from GDSI)
grid['drought_freq'] = join_grace_to_grid(
    PROC_GW / 'gdsi_drought_metrics_grid.nc', 'drought_freq_pct', grid)

# Recent SGWA mean (2019-2023)
grid['sgwa_recent'] = join_grace_to_grid(
    PROC_GW / 'gdsi_drought_metrics_grid.nc', 'sgwa_recent_mean', grid)

# GWSA trend (Sen's slope, mm/yr)
grid['gwsa_slope'] = join_grace_to_grid(
    PROC_GW / 'gwsa_trend_pixelwise.nc', 'sens_slope_mm_yr', grid)

# Proxy subsidence potential
grid['proxy_sub'] = join_grace_to_grid(
    PROC_SUB / 'proxy_subsidence_potential_grid.nc',
    'proxy_subsidence_potential', grid)

print('GRACE indicators joined:')
for col in ['drought_freq','sgwa_recent','gwsa_slope','proxy_sub']:
    print(f'  {col}: mean={grid[col].mean():.3f}, range=[{grid[col].min():.3f}, {grid[col].max():.3f}]')


In [ ]:
# 4. Piped water access proxy
# No direct data available. Proxy: inverse distance to nearest major urban centre
# (areas far from Accra, Kumasi, Cape Coast, Takoradi have less piped water access)
# Combined with low population density as co-indicator of lower pipe coverage

# Major cities with piped water supply in Ghana (WGS84)
cities_wgs = {
    'Accra':     (-0.186964, 5.603717),
    'Tema':      (0.016667,  5.670000),
    'Kumasi':    (-1.623000, 6.688000),
    'Cape Coast':(-1.246944, 5.104444),
    'Takoradi':  (-1.750000, 4.900000),
    'Koforidua': (-0.261111, 6.088056),
    'Ho':        (0.472222,  6.601111),
}

city_pts = np.array([(lat, lon) for lon, lat in cities_wgs.values()])
city_tree = cKDTree(city_pts)

grid_pts = np.column_stack([grid['centroid_lat'].values, grid['centroid_lon'].values])
dist_deg, _ = city_tree.query(grid_pts, k=1)
# Convert degrees to approximate km (1 degree ~111 km at equator)
grid['dist_to_city_km'] = dist_deg * 111.0

# PWA = inverse proxy: farther = less piped water access = higher pressure
# Normalise 0-1: 0 = near city (good access), 1 = far (poor access)
d_min = grid['dist_to_city_km'].min()
d_max = grid['dist_to_city_km'].max()
grid['pwa_proxy'] = (grid['dist_to_city_km'] - d_min) / (d_max - d_min + 1e-9)

print(f'Distance to nearest city: {grid["dist_to_city_km"].min():.1f} – {grid["dist_to_city_km"].max():.1f} km')
print(f'PWA proxy (0=good access, 1=poor access): mean={grid["pwa_proxy"].mean():.3f}')


In [ ]:
# 5. Normalise all indicators using min-max (0-1)

def minmax_norm(series, invert=False):
    s = pd.to_numeric(series, errors='coerce')
    mn, mx = s.min(), s.max()
    n = (s - mn) / (mx - mn + 1e-9)
    n = n.clip(0, 1)
    return 1 - n if invert else n

# GWSA decline: negative slope = declining GW = higher pressure
# Invert so that MORE NEGATIVE slope → higher UGPI score
grid['gwsa_decline'] = (-grid['gwsa_slope']).clip(lower=0)  # only declining pixels

indicators = {
    'BD':   ('builtup_density_2020',  False, 'Building density 2020'),
    'PD':   ('pop_density_km2',       False, 'Population density (people/km²)'),
    'BUGR': ('builtup_growth_pct',    False, 'Built-up growth rate 2000-2025 (%)'),
    'DRT':  ('drought_freq',          False, 'Drought frequency (% months SGWA<-1)'),
    'GW':   ('gwsa_decline',          False, 'GW storage decline (neg. trend)'),
    'PWA':  ('pwa_proxy',             False, 'Piped water access proxy (inverse)'),
}

norm_cols = {}
print('Normalised indicator statistics:')
print(f'{"Indicator":<6} {"Raw min":>10} {"Raw max":>10} {"Norm mean":>10} {"Label"}')
for abbr, (col, inv, label) in indicators.items():
    n_col = f'n_{abbr}'
    grid[n_col] = minmax_norm(grid[col], invert=inv)
    norm_cols[abbr] = n_col
    print(f'{abbr:<6} {grid[col].min():>10.3f} {grid[col].max():>10.3f} '
          f'{grid[n_col].mean():>10.3f}  {label}')


In [ ]:
# 6. Compute UGPI — equal weights baseline
n = len(indicators)
w_equal = {abbr: 1/n for abbr in indicators}

grid['ugpi_equal'] = sum(
    w_equal[abbr] * grid[norm_cols[abbr]]
    for abbr in indicators
)

# Classify into 5 tiers
bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.001]
labels = ['Very low', 'Low', 'Moderate', 'High', 'Very high']
grid['ugpi_class'] = pd.cut(grid['ugpi_equal'], bins=bins, labels=labels,
                            right=True, include_lowest=True)

print(f'UGPI (equal weights) — {len(grid)} cells')
print(f'  Range:  {grid["ugpi_equal"].min():.3f} – {grid["ugpi_equal"].max():.3f}')
print(f'  Mean:   {grid["ugpi_equal"].mean():.3f}')
print(f'  Median: {grid["ugpi_equal"].median():.3f}')
print()
print('Class distribution:')
for cls, cnt in grid['ugpi_class'].value_counts().sort_index().items():
    pct = cnt / len(grid) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls:12s}: {cnt:4d} ({pct:4.1f}%) {bar}')


In [ ]:
# 7. Expert-informed weighting scenario
# Emphasise urban pressure (BD, PD, BUGR) and drought stress (DRT, GW)
# De-weight PWA proxy (least validated indicator)
w_expert = {
    'BD':   0.20,
    'PD':   0.20,
    'BUGR': 0.20,
    'DRT':  0.15,
    'GW':   0.15,
    'PWA':  0.10,
}
assert abs(sum(w_expert.values()) - 1.0) < 1e-9, 'Weights must sum to 1'

grid['ugpi_expert'] = sum(
    w_expert[abbr] * grid[norm_cols[abbr]]
    for abbr in indicators
)

# Urban-only weighting (BD + PD + BUGR only)
grid['ugpi_urban_only'] = (
    grid[norm_cols['BD']] + grid[norm_cols['PD']] + grid[norm_cols['BUGR']]
) / 3

# Hydro-only weighting (DRT + GW)
grid['ugpi_hydro_only'] = (
    grid[norm_cols['DRT']] + grid[norm_cols['GW']]
) / 2

# Correlation between scenarios
scenarios = ['ugpi_equal','ugpi_expert','ugpi_urban_only','ugpi_hydro_only']
print('Spearman correlation between UGPI weighting scenarios:')
corr_df = pd.DataFrame(index=scenarios, columns=scenarios, dtype=float)
# Align on common non-null rows before computing correlation
common_idx = grid[scenarios].dropna().index
for s1 in scenarios:
    for s2 in scenarios:
        r, _ = spearmanr(grid.loc[common_idx, s1], grid.loc[common_idx, s2])
        corr_df.loc[s1, s2] = round(r, 3)
print(corr_df.to_string())


In [ ]:
# 8. Top-pressure cells and municipality stats

# Top 20 cells by UGPI (equal weights)
top20 = grid.nlargest(20, 'ugpi_equal')[[
    'cell_id','centroid_lat','centroid_lon','ugpi_equal','ugpi_class',
    'builtup_density_2020','pop_density_km2','builtup_growth_pct',
    'drought_freq','gwsa_slope'
]].reset_index(drop=True)
print('Top 20 high-pressure cells (UGPI equal weights):')
print(top20[['cell_id','centroid_lat','centroid_lon','ugpi_equal','ugpi_class',
             'builtup_density_2020','pop_density_km2']].to_string(index=False))

top20.to_csv(TAB_DIR / 'ugpi_top20_cells.csv', index=False)


In [ ]:
# 9. Sensitivity analysis — vary weights ±50% one at a time
base_weights = w_equal.copy()
sensitivity = []

for perturb_ind in indicators:
    for direction, factor in [('increase', 1.5), ('decrease', 0.5)]:
        w_test = base_weights.copy()
        w_test[perturb_ind] *= factor
        # Renormalise so weights sum to 1
        total = sum(w_test.values())
        w_test = {k: v/total for k, v in w_test.items()}
        ugpi_test = sum(w_test[abbr] * grid[norm_cols[abbr]] for abbr in indicators)
        r, _ = spearmanr(grid['ugpi_equal'].dropna(), ugpi_test.dropna())
        sensitivity.append({
            'perturbed_indicator': perturb_ind,
            'direction': direction,
            'weight_factor': factor,
            'spearman_r_vs_base': round(r, 4),
            'rank_stability': 'stable' if r >= 0.95 else 'moderate' if r >= 0.85 else 'sensitive',
        })

df_sens = pd.DataFrame(sensitivity)
df_sens.to_csv(TAB_DIR / 'ugpi_sensitivity_analysis.csv', index=False)
print('Sensitivity analysis (Spearman r vs equal-weight UGPI):')
print(df_sens.pivot(index='perturbed_indicator', columns='direction',
                    values='spearman_r_vs_base').to_string())


In [ ]:
# 10. Save all outputs
# Drop geometry for CSV
ugpi_cols = ['cell_id','centroid_lat','centroid_lon',
             'n_BD','n_PD','n_BUGR','n_DRT','n_GW','n_PWA',
             'ugpi_equal','ugpi_expert','ugpi_urban_only','ugpi_hydro_only','ugpi_class']
grid[ugpi_cols].to_csv(PROC_IDX / 'ugpi_grid10km.csv', index=False)

# Save GeoPackage with all columns
grid.to_file(PROC_IDX / 'ugpi_grid10km.gpkg', driver='GPKG')

print('Saved CSV and GeoPackage')


In [ ]:
# 11. Figure: normalised indicator maps (2x3 panel)
import cartopy.crs as ccrs
import cartopy.feature as cfeature

ind_labels = {
    'n_BD':   'A  Building Density',
    'n_PD':   'B  Population Density',
    'n_BUGR': 'C  Built-Up Growth Rate',
    'n_DRT':  'D  Drought Frequency',
    'n_GW':   'E  GW Storage Decline',
    'n_PWA':  'F  Poor Piped Water Access',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                          subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()
fig.suptitle('UGPI Component Indicators — Coastal Ghana (normalised 0–1)',
             fontsize=12, fontweight='bold')

for ax, (col, title) in zip(axes, ind_labels.items()):
    sc = ax.scatter(grid['centroid_lon'], grid['centroid_lat'],
                   c=grid[col], cmap='YlOrRd', s=10, vmin=0, vmax=1, alpha=0.85,
                   transform=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=0.6)
    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)
    ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
    ax.set_title(title, loc='left', fontsize=9)
    gl = ax.gridlines(draw_labels=False, lw=0.2, alpha=0.4)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=mcolors.Normalize(0, 1))
sm.set_array([])
fig.colorbar(sm, cax=cbar_ax, label='Normalised indicator value (0–1)')

fig.tight_layout(rect=[0, 0, 0.91, 0.96])
fig.savefig(FIG_DIR / 'ugpi_indicator_maps.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'ugpi_indicator_maps.png')


In [ ]:
# 12. Figure: UGPI equal-weight map with top-pressure zones
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Patch

fig = plt.figure(figsize=(11, 10))
ax  = fig.add_subplot(111, projection=ccrs.PlateCarree())

cmap = plt.cm.get_cmap('RdYlGn_r', 5)
bounds = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
norm   = mcolors.BoundaryNorm(bounds, cmap.N)

sc = ax.scatter(grid['centroid_lon'], grid['centroid_lat'],
               c=grid['ugpi_equal'], cmap=cmap, norm=norm,
               s=25, alpha=0.85, transform=ccrs.PlateCarree())

# Overlay top 5% cells as larger markers
top5pct = grid[grid['ugpi_equal'] >= grid['ugpi_equal'].quantile(0.95)]
ax.scatter(top5pct['centroid_lon'], top5pct['centroid_lat'],
           facecolors='none', edgecolors='black', s=80, lw=1.5,
           transform=ccrs.PlateCarree(), zorder=5, label='Top 5% pressure')

ax.add_feature(cfeature.COASTLINE, lw=0.8)
ax.add_feature(cfeature.BORDERS, lw=0.5, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)
ax.add_feature(cfeature.RIVERS, edgecolor='#4292c6', lw=0.4)

# Key city labels
for city, (lon_c, lat_c) in {
    'Accra': (-0.19, 5.55), 'Tema': (0.00, 5.67),
    'Takoradi': (-1.75, 4.90), 'Cape Coast': (-1.25, 5.12),
    'Kumasi': (-1.62, 6.69), 'Ho': (0.47, 6.60)
}.items():
    ax.plot(lon_c, lat_c, 'ks', ms=5, transform=ccrs.PlateCarree(), zorder=6)
    ax.text(lon_c + 0.06, lat_c + 0.06, city, fontsize=7.5,
            transform=ccrs.PlateCarree(), zorder=6)

cbar = fig.colorbar(sc, ax=ax, shrink=0.65, pad=0.02, ticks=bounds)
cbar.set_ticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'])
cbar.set_label('UGPI (equal weights, 0–1)', fontsize=10)
ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
ax.set_title('Urban Groundwater Pressure Index (UGPI) — Coastal Ghana\n'
             'Equal-weight composite: building density + population + urban growth\n'
             '+ drought frequency + GW decline + piped-water access proxy',
             fontsize=10)
ax.legend(fontsize=9, loc='upper right')
gl = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

fig.tight_layout()
fig.savefig(FIG_DIR / 'ugpi_map.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'ugpi_map.png')


In [ ]:
# 13. Stage 9 summary
print('=' * 60)
print('Stage 9 complete')
outputs = [
    PROC_IDX / 'ugpi_grid10km.gpkg',
    PROC_IDX / 'ugpi_grid10km.csv',
    TAB_DIR  / 'ugpi_top20_cells.csv',
    TAB_DIR  / 'ugpi_sensitivity_analysis.csv',
    FIG_DIR  / 'ugpi_indicator_maps.png',
    FIG_DIR  / 'ugpi_map.png',
]
for p in outputs:
    tick = 'OK' if Path(p).exists() else 'MISSING'
    print(f'  [{tick}]  {Path(p).name}')

print()
print(f'UGPI stats (equal weights):')
print(f'  Mean:  {grid["ugpi_equal"].mean():.3f}')
print(f'  Max:   {grid["ugpi_equal"].max():.3f}')
print(f'  High+Very high cells: {(grid["ugpi_class"].isin(["High","Very high"])).sum()}')
print()
print('Sensitivity: all scenarios Spearman r >= 0.95 = stable')
min_r = df_sens['spearman_r_vs_base'].min()
print(f'  Minimum Spearman r across all perturbations: {min_r:.3f}')
print(f'  Overall stability: {"stable" if min_r >= 0.95 else "review needed"}')
